<a href="https://colab.research.google.com/github/Maryam-Shile/Intuitive_pythonista_projects/blob/main/Bank_account_predicition_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import GradientBoostingClassifier
import warnings
warnings.filterwarnings('ignore')

In [3]:
!unzip /content/financial-inclusion-in-africa20250311-22142-nbnoiv.zip -d /content/financial_inclusion

Archive:  /content/financial-inclusion-in-africa20250311-22142-nbnoiv.zip
  inflating: /content/financial_inclusion/manifest-e268f67161c155de502276443b494f7c20250311-22142-1qdfioo.json  
  inflating: /content/financial_inclusion/Train.csv  
  inflating: /content/financial_inclusion/Test.csv  
  inflating: /content/financial_inclusion/VariableDefinitions.csv  
  inflating: /content/financial_inclusion/SampleSubmission.csv  
  inflating: /content/financial_inclusion/StarterNotebook.ipynb  


In [4]:
train = pd.read_csv('/content/financial_inclusion/Train.csv')
test = pd.read_csv('/content/financial_inclusion/Test.csv')

In [5]:
train['uniqueid'] = train['uniqueid'] + ' x ' + train['country']
test['uniqueid'] = test['uniqueid'] + ' x ' + test['country']

In [6]:
age_boundaries = [0, 18, 25, 35, 45, 55, 65, np.inf]
labels = ['minors', 'gen_z', 'young_professionals', 'family_formers', 'mature_adults', 'pre_retirement', 'seniors']
train['age_group'] = pd.cut(x = train['age_of_respondent'], bins = age_boundaries, labels = labels, right = False)
test['age_group'] = pd.cut(x = train['age_of_respondent'], bins = age_boundaries, labels = labels, right = False)

In [13]:
X = train.drop(columns = ['uniqueid', 'bank_account', 'age_of_respondent'])
y = train['bank_account']

y = y.map({'Yes': 1, 'No': 0})

zindi_test = test.drop(columns = ['uniqueid', 'age_of_respondent'])


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

y

,bank_account
0,1
1,0
2,1
3,0
4,0
...,...
23519,0
23520,0
23521,0
23522,0


In [15]:
cat_col = ['country', 'year', 'location_type',
       'cellphone_access', 'gender_of_respondent', 'relationship_with_head', 'marital_status',
       'education_level', 'job_type', 'age_group']
num_col = ['household_size']

preprocessor = ColumnTransformer(transformers = [
    ('cat', OneHotEncoder(drop = 'first', sparse_output = False), cat_col),
    ('num', RobustScaler(), num_col)
], remainder = 'passthrough')

pipeline = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('classifier', LGBMClassifier(random_state = 42))
])

param_grid = {'classifier__num_leaves': [57, 75, 81, 87],
              'classifier__max_depth': [12, 14, 16],
              'classifier__min_data_in_leaf': [20, 50, 100],
              'classifier__learning_rate': [0.01],
              'classifier__is_unbalance': [True]}

grid = GridSearchCV(estimator = pipeline, param_grid = param_grid, cv = 5, scoring ='f1', n_jobs = -1, verbose = 1)

grid.fit(X_train, y_train)

print(f'Best Parameters: {grid.best_params_}')
print(f'Best CV Score: {grid.best_score_:.4f}')
print(f'Test Score: {grid.score(X_test, y_test):.4f}')



Fitting 5 folds for each of 36 candidates, totalling 180 fits
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Info] Number of positive: 2670, number of negative: 16149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001962 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 88
[LightGBM] [Info] Number of data points in the train set: 18819, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.141878 -> initscore=-1.799780
[LightGBM] [Info] Start training from score -1.799780
Best Parameters: {'classifier__is_unbalance': True, 'classifier__learning_rate': 0.01, 'classifier__max_depth': 14, 'classifier__min_data_i

In [16]:
best = grid.best_estimator_

best.fit(X, y)

y_pred = best.predict(zindi_test)

[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Info] Number of positive: 3312, number of negative: 20212
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001652 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 88
[LightGBM] [Info] Number of data points in the train set: 23524, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.140792 -> initscore=-1.808724
[LightGBM] [Info] Start training from score -1.808724
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50


In [17]:
print(f'zindi test set: {len(zindi_test)}')
print(f'y_pred: {len(y_pred)}')

zindi test set: 10086
y_pred: 10086


In [19]:
zindi_test.columns

Index(['country', 'year', 'location_type', 'cellphone_access',
       'household_size', 'gender_of_respondent', 'relationship_with_head',
       'marital_status', 'education_level', 'job_type', 'age_group'],
      dtype='object')

In [20]:
final_submission = pd.DataFrame({'unique_id': test['uniqueid'],
                                'bank_account': y_pred })
final_submission

,unique_id,bank_account
0,uniqueid_6056 x Kenya,1
1,uniqueid_6060 x Kenya,1
2,uniqueid_6065 x Kenya,0
3,uniqueid_6072 x Kenya,0
4,uniqueid_6073 x Kenya,0
...,...,...
10081,uniqueid_2998 x Uganda,0
10082,uniqueid_2999 x Uganda,0
10083,uniqueid_3000 x Uganda,0
10084,uniqueid_3001 x Uganda,0


In [21]:
final_submission.to_csv('second_submission.csv', index = False)